In [0]:
from pyspark.sql import functions as F

# ============================================================
# FACT_PEDIDOS - GOLD (VOLUME / DELTA PATH)
# ============================================================

pedidos = spark.read.format("delta").load(
    "/Volumes/workspace/default/sources/delta/silver_pedidos"
)

itens = spark.read.format("delta").load(
    "/Volumes/workspace/default/sources/delta/silver_itens"
)

fact_pedidos = (
    pedidos.alias("p")
    .join(itens.alias("i"), "order_id", "left")
    .select(
        "p.order_id",
        "p.customer_code",
        "p.seller_id",
        "p.order_date",
        "p.status_order",

        "i.product_code",
        "i.item_seq",

        F.col("i.quantity").cast("int").alias("quantity"),
        F.col("i.unit_price").cast("double").alias("unit_price"),
        F.col("i.total_item").cast("double").alias("total_item"),

        (F.col("i.quantity").cast("int") * F.col("i.unit_price").cast("double")).alias("gross_revenue")
    )
)

fact_pedidos.write.format("delta").mode("overwrite").save(
    "/Volumes/workspace/default/sources/delta/fact_pedidos"
)

print("✔ fact_pedidos criada com sucesso")

✔ fact_pedidos criada com sucesso


In [0]:
from pyspark.sql import functions as F

clientes = spark.read.format("delta").load(
    "/Volumes/workspace/default/sources/delta/silver_clientes"
)

dim_cliente = (
    clientes
    .select(
        "customer_id",
        "nome_cliente",
        "segmento",
        "porte",
        "cidade",
        "estado",
        "status_cliente",
        "data_cadastro"
    )
    .dropDuplicates(["customer_id"])
)

dim_cliente.write.format("delta").mode("overwrite").save(
    "/Volumes/workspace/default/sources/delta/dim_cliente"
)

print("✔ dim_cliente criada")

✔ dim_cliente criada


In [0]:
from pyspark.sql import functions as F

produtos = spark.read.format("delta").load(
    "/Volumes/workspace/default/sources/delta/silver_produtos"
)

dim_produto = (
    produtos
    .select(
        F.col("product.product_id").alias("product_id"),
        F.col("product.name").alias("product_name"),
        F.col("product.category").alias("category"),
        F.col("product.subcategory").alias("subcategory"),
        F.col("product.status").alias("status"),
        F.col("family"),
        F.col("currency"),
        F.col("pricing.list_price").cast("double").alias("list_price")
    )
    .dropDuplicates(["product_id"])
)

dim_produto.write.format("delta").mode("overwrite").save(
    "/Volumes/workspace/default/sources/delta/dim_produto"
)

print("✔ dim_produto criada")

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4935824793384650>, line 22
      3 produtos = spark.read.format("delta").load(
      4     "/Volumes/workspace/default/sources/delta/silver_produtos"
      5 )
      7 dim_produto = (
      8     produtos
      9     .select(
   (...)
     19     .dropDuplicates(["product_id"])
     20 )
---> 22 dim_produto.write.format("delta").mode("overwrite").save(
     23     "/Volumes/workspace/default/sources/delta/dim_produto"
     24 )
     26 print("✔ dim_produto criada")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:703, in DataFrameWriter.save(self, path, format, mode, partitionBy, **options)
    701     self.format(format)
    702 self._write.path = path
--> 703 _, _, ei = self._spark.client.execute_command(
    704     self._write.command(self._spark.client), self._write.observations


In [0]:
vendedores = spark.read.format("delta").load(
    "/Volumes/workspace/default/sources/delta/silver_vendedores"
)

dim_vendedor = (
    vendedores
    .select(
        "seller_id",
        "seller_name",
        "status",
        "regional_code",
        "hire_date",
        "canal_id"
    )
    .dropDuplicates(["seller_id"])
)

dim_vendedor.write.format("delta").mode("overwrite").save(
    "/Volumes/workspace/default/sources/delta/dim_vendedor"
)

print("✔ dim_vendedor criada")

✔ dim_vendedor criada


In [0]:
regioes = spark.read.format("delta").load(
    "/Volumes/workspace/default/sources/delta/silver_regioes"
)

dim_regiao = (
    regioes
    .select(
        "regional_code",
        "regional_name",
        "state",
        "manager_name",
        "active_flag"
    )
    .dropDuplicates(["regional_code"])
)

dim_regiao.write.format("delta").mode("overwrite").save(
    "/Volumes/workspace/default/sources/delta/dim_regiao"
)

print("✔ dim_regiao criada")

✔ dim_regiao criada


In [0]:
canais = spark.read.format("delta").load(
    "/Volumes/workspace/default/sources/delta/silver_canais"
)

dim_canal = (
    canais
    .select(
        "id_canal",
        "nome_canal",
        "tipo_canal",
        "ativo",
        "observacao"
    )
    .dropDuplicates(["id_canal"])
)

dim_canal.write.format("delta").mode("overwrite").save(
    "/Volumes/workspace/default/sources/delta/dim_canal"
)

print("✔ dim_canal criada")

✔ dim_canal criada


In [0]:
from pyspark.sql import functions as F

produtos = spark.read.format("delta").load(
    "/Volumes/workspace/default/sources/delta/silver_produtos"
)

dim_produto = (
    produtos
    .select(
        "product_id",
        "product_name",
        "category",
        "subcategory",
        "status",
        "family",
        "currency",
        F.col("list_price").cast("double").alias("list_price"),
        "updated_at"
    )
    .dropDuplicates(["product_id"])
)

dim_produto.write.format("delta").mode("overwrite").save(
    "/Volumes/workspace/default/sources/delta/dim_produto"
)

print("✔ dim_produto criada com sucesso")

✔ dim_produto criada com sucesso


In [0]:
entregas_bronze = spark.read.format("delta").load(
    "/Volumes/workspace/default/sources/delta/bronze_entregas"
)

entregas = (
    entregas_bronze
    .select("*")
    # aqui você reaplica sua normalização se quiser
)

entregas.write.format("delta").mode("overwrite").save(
    "/Volumes/workspace/default/sources/delta/silver_entregas"
)

print("✔ silver_entregas recriada")

✔ silver_entregas recriada


In [0]:
from pyspark.sql import functions as F

entregas = spark.read.format("delta").load(
    "/Volumes/workspace/default/sources/delta/silver_entregas"
)

# 🔥 explode JSON de timestamps
entregas = entregas.withColumn(
    "shipped_at",
    F.get_json_object("timestamps", "$.shipped_at")
).withColumn(
    "delivered_at",
    F.get_json_object("timestamps", "$.delivered_at")
)

dim_entrega = (
    entregas
    .select(
        F.col("order_ref").alias("delivery_id"),
        "carrier",
        "cost",
        "shipped_at",
        "delivered_at"
    )
    .dropDuplicates(["delivery_id"])
)

dim_entrega.write.format("delta").mode("overwrite").save(
    "/Volumes/workspace/default/sources/delta/dim_entrega"
)

print("✔ dim_entrega criada com sucesso")

✔ dim_entrega criada com sucesso


In [0]:
spark.sql("""
DROP TABLE IF EXISTS delta.`/Volumes/workspace/default/sources/delta/fact_pedidos_dim`
""")

DataFrame[]

# 🧠 Case de Engenharia de Dados - Modelo Analítico

## 📌 Visão Geral

Este projeto implementa uma arquitetura de dados em camadas (Bronze, Silver e Gold), transformando dados brutos em um modelo analítico pronto para consumo por BI.

---

## 🏗️ Arquitetura

- Bronze: ingestão bruta
- Silver: limpeza e padronização
- Gold: modelo estrela analítico

---

## ⭐ Modelo Final

Foi construído um modelo estrela contendo:

### Fact
- fact_pedidos (nível item)

### Dimensões
- dim_cliente
- dim_produto
- dim_vendedor
- dim_regiao
- dim_canal

---

## 📊 KPIs Disponíveis

- Receita total
- Volume de pedidos
- Ticket médio
- Receita por região
- Receita por canal
- Evolução temporal de vendas

---

## 🔧 Tecnologias

- PySpark
- Databricks
- Delta Lake

---

## 🧠 Decisões Técnicas

- Granularidade no nível de item de pedido
- Padronização de dados na camada Silver
- Modelo estrela para performance analítica
- Evitado enriquecimento excessivo na fact

---

## ⚠️ Limitações

- Algumas dimensões derivadas de campos categóricos do cliente
- Ausência de geolocalização detalhada
- Dados de canal inferidos em alguns casos

---

## 🚀 Resultado

O modelo final permite análises completas de performance comercial, operacional e logística com baixa complexidade para consumo em BI.